In [99]:
# Load all the libraries
import warnings
warnings.filterwarnings('ignore')

import sys
import pathlib
from pathlib import Path
import pickle
import os
import numpy as np
import pandas as pd
from netCDF4 import Dataset as NetCDFFile 
import import_ipynb
from datetime import date, datetime, timedelta
import datetime as dt
# import oceans.sw_extras as swe
import gsw
import matplotlib.pyplot as plt
import timezonefinder, pytz
tf = timezonefinder.TimezoneFinder()

# Import SOCA ligh model funtions
import import_ipynb
from FUNCTIONS.OTHER_FUNCTIONS import rad_date, rad_LC_hour, derive_ZPD,find_MLD
from FUNCTIONS.PAR_ADJUSTED_FUNCTION import PAR_ADJUSTED_profiles
from FUNCTIONS.ED380_ADJUSTED_FUNCTION import ED380_ADJUSTED_profiles
from FUNCTIONS.ED412_ADJUSTED_FUNCTION import ED412_ADJUSTED_profiles
from FUNCTIONS.ED490_ADJUSTED_FUNCTION import ED490_ADJUSTED_profiles

In [105]:
# Force reload updated functions from notebooks
import importlib
import sys

# Remove cached modules to force reload
modules_to_reload = [
    'FUNCTIONS.PAR_ADJUSTED_FUNCTION',
    'FUNCTIONS.ED380_ADJUSTED_FUNCTION', 
    'FUNCTIONS.ED412_ADJUSTED_FUNCTION',
    'FUNCTIONS.ED490_ADJUSTED_FUNCTION',
    'FUNCTIONS.OTHER_FUNCTIONS'
]

for module in modules_to_reload:
    if module in sys.modules:
        del sys.modules[module]

# Now reimport with fresh modules
from FUNCTIONS.OTHER_FUNCTIONS import rad_date, rad_LC_hour, derive_ZPD,find_MLD
from FUNCTIONS.PAR_ADJUSTED_FUNCTION import PAR_ADJUSTED_profiles
from FUNCTIONS.ED380_ADJUSTED_FUNCTION import ED380_ADJUSTED_profiles
from FUNCTIONS.ED412_ADJUSTED_FUNCTION import ED412_ADJUSTED_profiles
from FUNCTIONS.ED490_ADJUSTED_FUNCTION import ED490_ADJUSTED_profiles

print("Functions successfully reloaded!")

Functions successfully reloaded!


In [110]:
## The inputs from excel file a sample file (INPUTS/inputs.xlsx) 
## The output file name as DAC_WMO_CYCLE_ED490.txt format this can change as per users choice

#######################################################################################
# define output path (WHERE YOU WANT TO SAVE THE OUTPUT FILES)
# PAR_OUT_PATH = pathlib.Path('/Users/renoshpr/Documents/BLUE_CLOUD/SOCA_CHL_2020/NOTEBOOK/PAR_ADJUSTED_MODEL_2022/PAR_ADJUSTED_Extraction_2023_51_depths/Output')
curr_work_dir = Path.cwd()
PAR_OUT_PATH = (''.join([str(curr_work_dir),'/OUTPUTS/PAR']))
ED380_OUT_PATH = (''.join([str(curr_work_dir),'/OUTPUTS/ED380']))
ED412_OUT_PATH = (''.join([str(curr_work_dir),'/OUTPUTS/ED412']))
ED490_OUT_PATH = (''.join([str(curr_work_dir),'/OUTPUTS/ED490']))

#######################################################################################
# READ INPUTS FROM EXCEL FILE
#
input_file_name = (''.join([str(curr_work_dir),'/SAT_MATCHUPS/satellite_matchups.csv']))
Inputs=pd.read_csv(str(input_file_name))

# print(Inputs)

for i in np.arange(len(Inputs)):
    # if i!=86:
    #     continue
    
    SEALTAG_NC_FILE = Inputs.SEALTAG_NC_FILE[i]
    PROFILE = Inputs.PROFILE_NUM[i]
    RHO412 = Inputs.RRS412[i]
    RHO443 = Inputs.RRS443[i]
    RHO490 = Inputs.RRS490[i]
    RHO555 = Inputs.RRS555[i]
    RHO670 = Inputs.RRS670[i]
    PAR = Inputs.PAR[i]
    KD490 = Inputs.KD490[i]
    PAR_output_file = (''.join([str(PAR_OUT_PATH),'/',str(Inputs.TAG_ID[i]),'_',str(PROFILE),'_PAR.txt']))
    ED380_output_file = (''.join([str(ED380_OUT_PATH),'/',str(Inputs.TAG_ID[i]),'_',str(PROFILE),'_ED380.txt']))
    ED412_output_file = (''.join([str(ED412_OUT_PATH),'/',str(Inputs.TAG_ID[i]),'_',str(PROFILE),'_ED412.txt']))
    ED490_output_file = (''.join([str(ED490_OUT_PATH),'/',str(Inputs.TAG_ID[i]),'_',str(PROFILE),'_ED490.txt']))
    
    #######################################################################################
    
    if os.path.isfile(SEALTAG_NC_FILE): # check Argo file exist
        # LOAD THE SEALTAG file
        #######################################################################################
        # ADD the path of the SEALTAG NC file
        NC= NetCDFFile(str(SEALTAG_NC_FILE))
        # Read ProfileInfo excel file
        profile_info_file = SEALTAG_NC_FILE.replace('_PROCESSED.nc','_ProfileInfo.xlsx')
        profile_info = pd.read_excel(profile_info_file, sheet_name='General')
    
        #######################################################################################
        ##### EXTRACTING TEMP SAL PRES same depths FROM NC ARGO FILE ##########################
        #######################################################################################
        pres_1 = NC.variables['PRES'][PROFILE-1,:]
        temp_1 = NC.variables['TEMP'][PROFILE-1,:]
        sal_1 = NC.variables['PSAL'][PROFILE-1,:]
        pres2 = pres_1.filled(fill_value=np.nan)
        temp2 = temp_1.filled(fill_value=np.nan)
        sal2 = sal_1.filled(fill_value=np.nan)
        
        # LAT = float(NC.variables['LATITUDE'][PROFILE-1])
        # LON = float(NC.variables['LONGITUDE'][PROFILE-1])
        LAT = float(profile_info.Lat[PROFILE-1])
        LON = float(profile_info.Lon[PROFILE-1])
        
        ########################################################################################
        # Declare output depths
        #Output daepth values 0, 5 10, 15, ... 250
        depth_output = np.linspace(0,250,51)
        #######################################################################################

        pres3=pres2[pres2<=250] # Light profile computed if T/S have at least 10 good points in 0 to 250 m water column
        pres3=pres2[pres2<=250] # Light profile computed if T/S have at least 15 good points in 0 to 250 m water column
        pres4=pres2[pres2<=50] # Light profile computed if T/S have at least 5 good points in 0 to 50 m water column
        if (len(pres3)>=15) & (len(pres4)>=5): # selected T/S profiles qualified above criterias
            #######################################################################################
            ##### EXTRACTING MLD ##################################################################
            #######################################################################################               
            # Derive Density Sigma0 and MLD from Temperature and Salinity profile
            # No need to change anything in this section by the user
            SA = gsw.SA_from_SP(sal2, pres2, LON, LAT) # Abslute salinity from practical salinity
            CT = gsw.CT_from_t(SA, temp2, pres2) # Conservative Temperature from in-situ sea water temperature
            dens = gsw.density.sigma0(SA,CT) # Potential density
            MLD_1 = find_MLD(Z=pres2, dens=dens, threshold = 0.03)
            print('MLD is :', MLD_1,'m')
            
            # Read MLD from ProfileInfo excel file
            MLD_1 = -profile_info.MLD[PROFILE-1]
            print('MLD is :', MLD_1,'m')
    
            #######################################################################################
            ##### EXTRACTING DOY FROM NC ARGO FILE ################################################
            #######################################################################################
            JULD = NC.variables['JULD'][PROFILE-1]
            # #Get day, month, year from juld
            ref_date = datetime(1950, 1, 1)
            date_prof = ref_date + timedelta(days=float(JULD))
            year_prof = date_prof.strftime('%Y')
            month_prof = date_prof.strftime('%m')
            day_prof = date_prof.strftime('%d')
            DOY = int(date_prof.strftime('%j'))
            # print(date_prof)
            # No need to change anything in this section by the user
            sin_doy = np.sin(rad_date(DOY))
            cos_doy = np.cos(rad_date(DOY))

            ########################################################################################################
            # Extract local time from the nc file and convert into radiance and then transform into Sine and Cosine
            ########################################################################################################
            timezone_str = tf.certain_timezone_at(lat=LAT, lng=LON)
            timezone = pytz.timezone(timezone_str)
            # date_prof is in GMT time scale
            LC = date_prof + timezone.utcoffset(date_prof) # +
            GMT_Hour = date_prof.strftime('%H')
            LC_Hour = LC.strftime('%H') 

            GMT_Hour_Min = date_prof.strftime('%H:%M')
            LC_Hour_Min = LC.strftime('%H:%M')  
            LC_Hour_Deci = (int(LC_Hour_Min[0:2]) + int(LC_Hour_Min[3:])/60)

            # LC_hour = 12
            sin_LC_hour = np.sin(rad_LC_hour(LC_Hour_Deci))
            cos_LC_hour = np.cos(rad_LC_hour(LC_Hour_Deci))
            ####################################################################################### 
            #######################################################################################
            # No need to change anything in this section by the user
            # HOW to use this function
    
            if(not np.isnan(MLD_1) and not np.isnan(PAR) and not np.isnan(KD490)and
                not np.isnan(RHO412) and not np.isnan(RHO443) and not np.isnan(RHO490) and
                not np.isnan(RHO555) and not np.isnan(RHO670)):
                PAR_ADJUSTED=PAR_ADJUSTED_profiles(PAR=PAR, MLD=MLD_1, ZPD=derive_ZPD(float(KD490)), RHO_WN_412=RHO412, RHO_WN_443=RHO443,
                                                    RHO_WN_490=RHO490, RHO_WN_555=RHO555, RHO_WN_670=RHO670, sin_doy=sin_doy,
                                                    cos_doy=cos_doy, sin_LC_hour=sin_LC_hour, cos_LC_hour=cos_LC_hour,
                                                    temp=temp2, sal=sal2, depths_argo=pres2, depth_output=depth_output)[0]
                ED380_ADJUSTED=ED380_ADJUSTED_profiles(PAR=PAR, MLD=MLD_1, ZPD=derive_ZPD(float(KD490)), RHO_WN_412=RHO412, RHO_WN_443=RHO443,
                                                    RHO_WN_490=RHO490, RHO_WN_555=RHO555, RHO_WN_670=RHO670, sin_doy=sin_doy,
                                                    cos_doy=cos_doy, sin_LC_hour=sin_LC_hour, cos_LC_hour=cos_LC_hour,
                                                    temp=temp2, sal=sal2, depths_argo=pres2, depth_output=depth_output)[0]
                ED412_ADJUSTED=ED412_ADJUSTED_profiles(PAR=PAR, MLD=MLD_1, ZPD=derive_ZPD(float(KD490)), RHO_WN_412=RHO412, RHO_WN_443=RHO443,
                                                    RHO_WN_490=RHO490, RHO_WN_555=RHO555, RHO_WN_670=RHO670, sin_doy=sin_doy,
                                                    cos_doy=cos_doy, sin_LC_hour=sin_LC_hour, cos_LC_hour=cos_LC_hour,
                                                    temp=temp2, sal=sal2, depths_argo=pres2, depth_output=depth_output)[0]
                ED490_ADJUSTED=ED490_ADJUSTED_profiles(PAR=PAR, MLD=MLD_1, ZPD=derive_ZPD(float(KD490)), RHO_WN_412=RHO412, RHO_WN_443=RHO443,
                                                    RHO_WN_490=RHO490, RHO_WN_555=RHO555, RHO_WN_670=RHO670, sin_doy=sin_doy,
                                                    cos_doy=cos_doy, sin_LC_hour=sin_LC_hour, cos_LC_hour=cos_LC_hour,
                                                    temp=temp2, sal=sal2, depths_argo=pres2, depth_output=depth_output)[0]                   
    
            else:
                PAR_ADJUSTED = np.nan
                ED380_ADJUSTED = np.nan
                ED412_ADJUSTED = np.nan
                ED490_ADJUSTED = np.nan
        else: # If temp/Sal criteria fails
            PAR_ADJUSTED = np.nan
            ED380_ADJUSTED = np.nan
            ED412_ADJUSTED = np.nan
            ED490_ADJUSTED = np.nan
        # else: # Temp and Sal not in delayed mode
        #     PAR_ADJUSTED = np.nan
        #     ED380_ADJUSTED = np.nan
        #     ED412_ADJUSTED = np.nan
        #     ED490_ADJUSTED = np.nan
    else: # If there is no BGC Argo file
        PAR_ADJUSTED = np.nan
        ED380_ADJUSTED = np.nan
        ED412_ADJUSTED = np.nan
        ED490_ADJUSTED = np.nan
              
    # #########################################################################################
    # ##########       
    db_par = {'depth': depth_output, 'PAR_ADJUSTED': PAR_ADJUSTED} 
    db_par = pd.DataFrame(db_par)
    db_par= db_par.round({'depth': 0, 'PAR_ADJUSTED': 4}) # Output adjusted to 4 decimal places
    
    db_ed380 = {'depth': depth_output, 'ED380_ADJUSTED': ED380_ADJUSTED} 
    db_ed380 = pd.DataFrame(db_ed380)
    db_ed380 = db_ed380.round({'depth': 0, 'ED380_ADJUSTED': 5}) # Output adjusted to 5 decimal places
    
    db_ed412 = {'depth': depth_output, 'ED412_ADJUSTED': ED412_ADJUSTED} 
    db_ed412 = pd.DataFrame(db_ed412)
    db_ed412 = db_ed412.round({'depth': 0, 'ED412_ADJUSTED': 5}) # Output adjusted to 5 decimal places
    
    db_ed490 = {'depth': depth_output, 'ED490_ADJUSTED': ED490_ADJUSTED} 
    db_ed490 = pd.DataFrame(db_ed490)
    db_ed490 = db_ed490.round({'depth': 0, 'ED490_ADJUSTED': 5}) # Output adjusted to 5 decimal places    
    
    ###########################################################################################
    db_par.to_csv(PAR_output_file, sep=',', index= False)
    db_ed380.to_csv(ED380_output_file, sep=',', index= False)   
    db_ed412.to_csv(ED412_output_file, sep=',', index= False)   
    db_ed490.to_csv(ED490_output_file, sep=',', index= False)   
       
    print(i)
    
    # Plot PAR_ADJUSTED profile down to 100 m for checking
    # plt.figure()
    # plt.plot(NC.variables['LIGHT'][PROFILE-1,:], -NC.variables['PRES'][PROFILE-1,:], 'ro')
    # plt.plot(db_par['PAR_ADJUSTED'], -db_par['depth'], '-o')
    # plt.xlabel('PAR_ADJUSTED (Ein m-2 s-1)')
    # plt.ylabel('Depth (m)')
    # plt.title('PAR_ADJUSTED Profile')     
    # plt.ylim(-100,0)
    # plt.legend(['SealTag Measured Light','Modeled PAR'])
##############################################################################################

MLD is : 81.0 m
MLD is : 80.23694618315712 m
0
MLD is : 92.0 m
MLD is : 91.13406779668595 m
1
MLD is : 94.0 m
MLD is : 92.2202075718249 m
0
MLD is : 92.0 m
MLD is : 91.13406779668595 m
1
MLD is : 94.0 m
MLD is : 92.2202075718249 m
2
MLD is : 84.0 m
MLD is : 82.5961104055584 m
3
MLD is : 47.0 m
MLD is : 46.05116413018782 m
2
MLD is : 84.0 m
MLD is : 82.5961104055584 m
3
MLD is : 47.0 m
MLD is : 46.05116413018782 m
4
MLD is : 35.0 m
MLD is : 34.60368694726549 m
5
MLD is : 33.0 m
MLD is : 31.738207621244243 m
4
MLD is : 35.0 m
MLD is : 34.60368694726549 m
5
MLD is : 33.0 m
MLD is : 31.738207621244243 m
6
MLD is : 34.0 m
MLD is : 32.931691839329744 m
7
MLD is : 29.0 m
MLD is : 28.409364963987965 m
6
MLD is : 34.0 m
MLD is : 32.931691839329744 m
7
MLD is : 29.0 m
MLD is : 28.409364963987965 m
8
MLD is : 10.0 m
MLD is : 13.348366728496572 m
9
MLD is : 34.0 m
MLD is : 29.452717510861145 m
8
MLD is : 10.0 m
MLD is : 13.348366728496572 m
9
MLD is : 34.0 m
MLD is : 29.452717510861145 m
10
MLD is

KeyboardInterrupt: 